In [30]:
import requests
import time
import pandas as pd
from pathlib import Path

In [31]:
# CVR API Base URL
BASE_URL = "https://cvrapi.dk/api"

# Default params for the API
# IMPORTANT: Provide a User-Agent identifying your application to avoid being blocked
headers = {
    "User-Agent": "DTU MSc HCAI - Project Social Data Analysis - Educational Use - Edgar Fabregat s242781@dtu.dk"
}

In [32]:
companies_data = pd.read_csv("../data/companies/full_list_companies.csv", sep=',', encoding='utf-8')
companies_list = companies_data['Company'].tolist()

In [33]:
#companies = companies_list[:36] # Done
#companies = companies_list[36:86] # Done
#companies = companies_list[86:136] # Done
companies = companies_list[136:] 
#companies = companies_list[150:]

In [34]:
# List of companies to search for
results = []

for company in companies:
    print(f"Searching for: {company}")
    params = {
        "country": "dk",
        "search": company
    }
    
    response = requests.get(BASE_URL, params=params, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        if "error" in data:
            if data['error'] == 'QUOTA_EXCEEDED':
                print("  API quota exceeded. Stopping further requests.")
                break
            print(f"  Error for {company}: {data['error']}")
        else:
            results.append(data)
            print(f"  Success: Found {data.get('name', 'Unknown')}, CVR: {data.get('vat', 'Unknown')}")
    else:
        print(f"  Failed with status code: {response.status_code}")
        
    # Be nice to the API - add a short delay
    time.sleep(4)

print(f"\nFetched data for {len(results)} companies.")

Searching for: SIEMENS GAMESA RENEWABLE ENERGY
  Success: Found Siemens Gamesa Renewable Energy A/S, CVR: 76486212
Searching for: SILKEBORG Kommune
  Success: Found Silkeborg Kommune, CVR: 29189641
Searching for: SINAL
  Success: Found Sinal A/S, CVR: 36423544
Searching for: SKANDERBORG Kommune
  Success: Found Skanderborg kommune, CVR: 29189633
Searching for: SPAREKASSEN DANMARK
  Success: Found Sparekassen Danmark, CVR: 64806815
Searching for: SPAREKASSEN KRONJYLLAND
  Success: Found SPAREKASSEN KRONJYLLAND, CVR: 17912828
Searching for: STARK GROUP
  Success: Found STARK Group A/S, CVR: 41952725
Searching for: STIBO SYSTEMS
  Success: Found STIBO SYSTEMS A/S, CVR: 35822690
Searching for: SVENDBORG KOMMUNE
  Success: Found Svendborg Kommune, CVR: 29189730
Searching for: SYSTEMATIC
  Success: Found SYSTEMATIC A/S, CVR: 78834412
Searching for: Salling Group
  Success: Found Salling Group A/S, CVR: 35954716
Searching for: Sparekassen Danmark
  Success: Found Sparekassen Danmark, CVR: 648

In [35]:
data_rows = []
index_list = []
keys = ['name', 'address', 'zipcode', 'city', 'startdate', 'enddate', 'employees']
columns = ['CompanyVat', 'CompanyName', 'UnitName', 'UnitAddress', 'UnitZipcode', 'UnitCity', 'UnitStartdate', 'UnitEnddate', 'UnitEmployees']

for res in results:
    for unit in res.get('productionunits', []):
        pno = unit.get('pno', None)
        values = [res['vat'], res['name']] + [unit.get(key, None) for key in keys]
        
        data_rows.append(values)
        index_list.append(pno)

# Create DataFrame
df_units = pd.DataFrame(data_rows, columns=columns, index=index_list)
#df_units.set_index('pno', inplace=True)
df_units.index.name = 'pno'

#display(df_units.head())

In [36]:
# Save to CSV
output_dir = Path("../data/CVR_API")
output_dir.mkdir(parents=True, exist_ok=True)

# Check if output file exists, if so, appendt to the file, otherwise create a new one
output_file = output_dir / "2_cvr_production_units.csv"
if output_file.exists():
    df_units.to_csv(output_file, mode='a', header=False, index=False, encoding='utf-8')
else:
    df_units.to_csv(output_file, index=False, encoding='utf-8')



In [37]:
# Save results from API to JSON
df_cvr = pd.DataFrame(results)

output_json_file = output_dir / "2_cvr_production_units.json"
if output_json_file.exists():
    df_cvr.to_json(output_json_file, orient='records', indent=4, mode='a')
else:
    df_cvr.to_json(output_json_file, orient='records', indent=4)

In [ ]:
# # Convert results to a pandas DataFrame and save to resources folder
# if results:
#     df_cvr = pd.DataFrame(results)
    
#     # Ensure resources directory exists
#     resources_dir = Path("../resources")
#     resources_dir.mkdir(parents=True, exist_ok=True)
    
#     output_path = resources_dir / "cvr_data.json"
    
#     # Save as JSON (preserves nested dictionary structures better than CSV)
#     df_cvr.to_json(output_path, orient="records", indent=4)
#     print(f"Saved corporate data to {output_path}")
    
#     # Also save a simplified CSV version for easier direct viewing
#     cols_to_save = ['vat', 'name', 'address', 'zipcode', 'city', 'phone', 'email', 'employees']
#     # Keep only available columns
#     available_cols = [col for col in cols_to_save if col in df_cvr.columns]
    
#     csv_out_path = resources_dir / "cvr_data.csv"
#     df_cvr[available_cols].to_csv(csv_out_path, index=False)
#     print(f"Saved simplified CSV to {csv_out_path}")
    
#     display(df_cvr[available_cols].head())
# else:
#     print("No data to save.")

### Combine DataFrames Edgar - Andrea - Uliyan

In [ ]:
# data sources
# "data/CVR_API/Andrea_cvr_production_units.json"
# "data/CVR_API/Uliyan_cvr_production_units.json"
import json
data_ = json.load(open("../data/CVR_API/Uliyan_cvr_production_units.json"))
data_[-1]

In [40]:
# Get .csv files from the data folder
main_data_file ="../data/CVR_API/cvr_production_units"
reading_files = ["../data/CVR_API/Uliyan_cvr_production_units","../data/CVR_API/Andrea_cvr_production_units", "../data/CVR_API/2_cvr_production_units"]
# csv_files = list(Path("../data/CVR_API").glob("*.csv"))
# csv_files.remove(main_data_file)
# print(csv_files)

#### Check Splits

In [41]:
companies_data = pd.read_csv("../data/companies/full_list_companies.csv", sep=',', encoding='utf-8')
companies_list = companies_data['Company'].tolist()

for file, rng_ in zip([main_data_file] + reading_files, [[0,36], [36,86], [86,136], [136, len(companies_list)]]):
    df1 = pd.read_json(file+".json", encoding='utf-8')
    print(f"File data: {file}\nCompany: {df1['name'][0]} - {df1['name'][len(df1)-1]} - Company List: {companies_list[rng_[0]]} - {companies_list[rng_[1]-1]}\n")


File data: ../data/CVR_API/cvr_production_units
Company: 15.JUNI FONDEN - DANSKE BANK A/S - Company List: 15.JUNI FONDEN - DANSKE BANK

File data: ../data/CVR_API/Uliyan_cvr_production_units
Company: DKT Holdings ApS - Køge Kommune - Company List: DKT HOLDINGS - KØGE KOMMUNE

File data: ../data/CVR_API/Andrea_cvr_production_units
Company: L'ORÉAL DANMARK A/S - SiccaDania Holding ApS - Company List: L'ORÉAL DANMARK - SICCADANIA

File data: ../data/CVR_API/2_cvr_production_units
Company: Siemens Gamesa Renewable Energy A/S - Kunstforeningen, Århus Universitetshospital, Risskov - Company List: SIEMENS GAMESA RENEWABLE ENERGY - ÅRHUS UNIVERSITETSHOSPITAL



In [46]:
type(json.load(open(reading_files[0]+".json", encoding='utf-8')))

list

#### Concat

In [ ]:
# Concat files with source tracking
complete_df = pd.read_csv(main_data_file+".csv", encoding='utf-8')

# Load JSON with source tracking
with open(main_data_file+".json", encoding='utf-8') as f:
    complete_json = json.load(f)

if not isinstance(complete_json, list):
    complete_json = [complete_json]

# Add source info to each record
for i, record in enumerate(complete_json):
    record['_source_file'] = main_data_file.split('/')[-1]
    record['_source_position'] = i

# Process additional files
for file in reading_files:
    df_csv = pd.read_csv(file+'.csv', encoding='utf-8')
    complete_df = pd.concat([complete_df, df_csv], ignore_index=True)

    with open(file+'.json', encoding='utf-8') as f:
        json_ = json.load(f)
    
    if not isinstance(json_, list):
        json_ = [json_]
    
    # Add source info to each record
    for i, record in enumerate(json_):
        record['_source_file'] = file.split('/')[-1]
        record['_source_position'] = i
    
    complete_json.extend(json_)

# Check duplicates in CSV
print("CSV Duplicates (CompanyVat + UnitName):")
print(complete_df.duplicated(subset=['CompanyVat', 'UnitName']).sum())
print()

# Check duplicates in JSON
df_complete_json = pd.DataFrame(complete_json)

duplicate_json = df_complete_json.duplicated(subset=['vat'], keep=False)
duplicated_records = df_complete_json[duplicate_json].sort_values('vat')

print(f"Duplicate entries in JSON data based on 'vat': {len(duplicated_records)}")
print("\nDuplicate details:")
for vat in duplicated_records['vat'].unique():
    print(f"\n{'='*80}")
    print(f"VAT: {vat}")
    print(f"Company: {duplicated_records[duplicated_records['vat'] == vat]['name'].iloc[0]}")
    print(f"{'='*80}")
    for idx, row in duplicated_records[duplicated_records['vat'] == vat].iterrows():
        print(f"  Source: {row['_source_file']}")
        print(f"  Position: {row['_source_position']}")
        print(f"  Name: {row['name']}")
        print()

In [ ]:
# Remove duplicates and save final JSON
df_deduplicated = df_complete_json.drop_duplicates(subset=['vat'], keep='first')

print(f"Original records: {len(df_complete_json)}")
print(f"Duplicates removed: {len(df_complete_json) - len(df_deduplicated)}")
print(f"Final records: {len(df_deduplicated)}")

# Convert back to list of records
final_json = df_deduplicated.to_dict('records')

# Save to final JSON file
final_output_file = output_dir / "cvr_production_units_final.json"
with open(final_output_file, 'w', encoding='utf-8') as f:
    json.dump(final_json, f, indent=4, ensure_ascii=False)

print(f"\nFinal deduplicated JSON saved to: {final_output_file}")

Original records: 169
Duplicates removed: 10
Final records: 159

Final deduplicated JSON saved to: ../data/CVR_API/cvr_production_units_final.json


In [70]:
# Remove duplicates and save final CSV
df_csv_deduplicated = complete_df.drop_duplicates(subset=['CompanyVat', 'CompanyName', 'UnitName', 'UnitAddress', 'UnitZipcode', 'UnitCity'], keep='first')

print(f"Original CSV records: {len(complete_df)}")
print(f"Duplicates removed: {len(complete_df) - len(df_csv_deduplicated)}")
print(f"Final CSV records: {len(df_csv_deduplicated)}")

# Save to final CSV file
final_csv_file = output_dir / "cvr_production_units_final.csv"
df_csv_deduplicated.to_csv(final_csv_file, index=False, encoding='utf-8')

print(f"\nFinal deduplicated CSV saved to: {final_csv_file}")

Original CSV records: 29938
Duplicates removed: 2379
Final CSV records: 27559

Final deduplicated CSV saved to: ../data/CVR_API/cvr_production_units_final.csv


In [71]:
# Compare companies list with deduplicated data
companies_df = pd.read_csv("../data/companies/full_list_companies.csv", sep=',', encoding='utf-8')
companies_list = companies_df['Company'].tolist()

# Get unique names from deduplicated JSON data
deduplicated_names = sorted(df_deduplicated['name'].unique().tolist())

print(f"Total companies in list: {len(companies_list)}")
print(f"Total unique names in deduplicated data: {len(deduplicated_names)}")
print(f"\n{'='*100}")

# Create comparison DataFrame
comparison_data = []
max_len = max(len(companies_list), len(deduplicated_names))

for i in range(max_len):
    company_name = companies_list[i] if i < len(companies_list) else "---"
    dedup_name = deduplicated_names[i] if i < len(deduplicated_names) else "---"
    
    # Mark if missing
    status = "✓" if dedup_name == "---" else ("✓" if company_name == dedup_name else "MISMATCH")
    
    comparison_data.append({
        'Index': i + 1,
        'Company List': company_name,
        'Deduplicated Data': dedup_name,
        'Status': status
    })

comparison_df = pd.DataFrame(comparison_data)

# Show full comparison
print("\nSide-by-side comparison:")
print(comparison_df.to_string(index=False))

# Show only missing companies
print(f"\n{'='*100}")
print("\nMISSING COMPANIES (in list but NOT in deduplicated data):")
missing = [c for c in companies_list if c not in deduplicated_names]
for i, company in enumerate(missing, 1):
    print(f"  {i}. {company}")

print(f"\nTotal missing: {len(missing)}")

Total companies in list: 161
Total unique names in deduplicated data: 159


Side-by-side comparison:
 Index                                   Company List                                                       Deduplicated Data   Status
     1                                 15.JUNI FONDEN                                                          15.JUNI FONDEN        ✓
     2                                       29190658                                                         365discount A/S MISMATCH
     3                                    365discount                                                 A.P. MØLLER - MÆRSK A/S MISMATCH
     4                            A.P. Møller - Mærsk                                                      AARHUS UNIVERSITET MISMATCH
     5                             AARHUS UNIVERSITET                                                           ACCENTURE A/S MISMATCH
     6                                      ACCENTURE                                    

In [74]:
# Remove unwanted companies from CSV, JSON, and companies list

# Companies to remove
csv_companies_to_remove = ['Forsvaret', 'HABITUS HOUSING AND DAY CARE', 'Nuuday', 'NUUDAY']
json_companies_to_remove = ['Jonathan Heavens Wolt Denmark', 'Edifice Housing and Projects A/S']

# Filter CSV - remove by CompanyName
df_csv_cleaned = df_csv_deduplicated[~df_csv_deduplicated['CompanyName'].isin(csv_companies_to_remove)].copy()

print(f"CSV before cleaning: {len(df_csv_deduplicated)}")
print(f"CSV after cleaning: {len(df_csv_cleaned)}")
print(f"CSV records removed: {len(df_csv_deduplicated) - len(df_csv_cleaned)}")

# Filter JSON - remove by name
df_json_cleaned = df_deduplicated[~df_deduplicated['name'].isin(json_companies_to_remove)].copy()

print(f"\nJSON before cleaning: {len(df_deduplicated)}")
print(f"JSON after cleaning: {len(df_json_cleaned)}")
print(f"JSON records removed: {len(df_deduplicated) - len(df_json_cleaned)}")

# Filter companies list - remove unwanted companies
df_companies_cleaned = companies_df[~companies_df['Company'].isin(csv_companies_to_remove)].copy()

print(f"\nCompanies list before cleaning: {len(companies_df)}")
print(f"Companies list after cleaning: {len(df_companies_cleaned)}")
print(f"Companies list records removed: {len(companies_df) - len(df_companies_cleaned)}")

# Save cleaned CSV file
final_csv_file_cleaned = output_dir / "cvr_production_units_final.csv"
df_csv_cleaned.to_csv(final_csv_file_cleaned, index=False, encoding='utf-8')
print(f"\nCleaned CSV saved to: {final_csv_file_cleaned}")

# Save cleaned JSON file
final_json_cleaned = df_json_cleaned.to_dict('records')
final_output_file_cleaned = output_dir / "cvr_production_units_final.json"
with open(final_output_file_cleaned, 'w', encoding='utf-8') as f:
    json.dump(final_json_cleaned, f, indent=4, ensure_ascii=False)
print(f"Cleaned JSON saved to: {final_output_file_cleaned}")

# Save cleaned companies list
companies_file_cleaned = Path("../data/companies/full_list_companies.csv")
df_companies_cleaned.to_csv(companies_file_cleaned, index=False, encoding='utf-8')
print(f"Cleaned companies list saved to: {companies_file_cleaned}")

CSV before cleaning: 27559
CSV after cleaning: 27559
CSV records removed: 0

JSON before cleaning: 159
JSON after cleaning: 157
JSON records removed: 2

Companies list before cleaning: 161
Companies list after cleaning: 157
Companies list records removed: 4

Cleaned CSV saved to: ../data/CVR_API/cvr_production_units_final.csv
Cleaned JSON saved to: ../data/CVR_API/cvr_production_units_final.json
Cleaned companies list saved to: ../data/companies/full_list_companies.csv


In [ ]:
forsvaret , HABITUS HOUSING AND DAY CARE Nuuday NUUDAY - 

Wolt --> Jonathan Heavens Wolt Denmark
Edifice Housing and Projects A/S